# Compartments


In [1]:
import sys
import warnings

warnings.filterwarnings("ignore")

sys.path.insert(0, "other")
from hic_common import *

N_WORKERS = 25   # coolers computed at the same time 
NPROC = 1        # threads inside 
#datasets and other settings in other/hic_common.py
print(f"{len(all_libraries())} libraries")

66 libraries


## Compute and cache E1 and saddles

In [2]:
tasks = build_tasks(NPROC)
print(f"{len(tasks)} coolers in total")

results = run_parallel(tasks, worker_compartments, N_WORKERS, "comp")

990 coolers in total
[1/990] comp skip         SRR13601502_full
[2/990] comp skip         SRR13601502_100000000_sample3
[3/990] comp skip         SRR13601502_100000000_sample2
[4/990] comp skip         SRR13601502_100000000_sample5
[5/990] comp skip         SRR13601502_100000000_sample1
[6/990] comp skip         SRR13601502_100000000_sample4
[7/990] comp skip         SRR13601502_10000000_sample11
[8/990] comp skip         SRR13601502_10000000_sample12
[9/990] comp skip         SRR13601502_10000000_sample14
[10/990] comp skip         SRR13601502_10000000_sample13
[11/990] comp skip         SRR13601502_10000000_sample16
[12/990] comp skip         SRR13601502_10000000_sample18
[13/990] comp skip         SRR13601502_10000000_sample10
[14/990] comp skip         SRR13601502_10000000_sample15
[15/990] comp skip         SRR13601502_10000000_sample1
[16/990] comp skip         SRR13601502_10000000_sample19
[17/990] comp skip         SRR13601502_10000000_sample17
[18/990] comp skip         SRR136

## Compare every subsample with its full-depth library

In [3]:
rows = []

for sample_name, srr in all_libraries():
    reference_id = f"{srr}_full"
    if not is_cached(eigs_path(sample_name, reference_id)):
        print(f"no full-depth E1 for {sample_name} / {srr}")
        continue
    reference = load_eigs(sample_name, reference_id)

    for entity in list_entities(sample_name, srr):
        entity_id = entity["entity_id"]
        if not (is_cached(eigs_path(sample_name, entity_id)) and
                is_cached(saddle_path(sample_name, entity_id))):
            continue

        eig = load_eigs(sample_name, entity_id)
        correlation = track_correlations(eig, reference, "E1")
        rows.append(dict(sample_name=sample_name, srr=srr, entity_id=entity_id,
                         role=entity["role"], depth=entity["depth"],
                         sample_idx=entity["sample_idx"],
                         comp_strength=load_saddle(sample_name, entity_id)["strength"],
                         E1_pearson=correlation["pearson"],
                         E1_spearman=correlation["spearman"],
                         E1_sign_agree=sign_agreement(eig, reference, "E1"),
                         n_bins=correlation["n_bins"]))

summary = pd.DataFrame(rows)
save_parquet(summary, DATASETS_DIR / "compartments_summary.parquet")

display(summary.head())

,sample_name,srr,entity_id,role,depth,sample_idx,comp_strength,E1_pearson,E1_spearman,E1_sign_agree,n_bins
0,U54-ESC4DN-DSG-DpnII-20190530-R1-T1,SRR13601502,SRR13601502_full,full,350000000,0,2.689783,1.000000,1.000000,1.000000,26253
1,U54-ESC4DN-DSG-DpnII-20190530-R1-T1,SRR13601502,SRR13601502_100000000_sample1,subsample,100000000,1,2.820104,0.945833,0.947042,0.934383,26243
2,U54-ESC4DN-DSG-DpnII-20190530-R1-T1,SRR13601502,SRR13601502_100000000_sample2,subsample,100000000,2,2.794131,0.954107,0.953277,0.936280,26240
3,U54-ESC4DN-DSG-DpnII-20190530-R1-T1,SRR13601502,SRR13601502_100000000_sample3,subsample,100000000,3,2.810548,0.959555,0.960728,0.939827,26241
4,U54-ESC4DN-DSG-DpnII-20190530-R1-T1,SRR13601502,SRR13601502_100000000_sample4,subsample,100000000,4,2.803495,0.955955,0.955117,0.938003,26243


In [4]:
def saddle_fixed_path(sample_name, entity_id):
    return COMPARTMENT_CACHE / sample_name / f"{entity_id}.saddle_fullE1.npz"


def worker_saddle_fixed(task):
    '''Saddle of one subsample, ranked by the full-depth E1 of the same library.'''
    sample_name, srr, entity = task["sample_name"], task["srr"], task["entity"]
    out = saddle_fixed_path(sample_name, entity["entity_id"])
    if is_cached(out):
        return ("skip", entity["entity_id"])
    try:
        if not ensure_weights(entity["path"], RES_COMPARTMENTS):
            return ("no weights", entity["entity_id"])
        clr = open_clr(entity["path"], RES_COMPARTMENTS)
        view = make_view(clr)
        full_eig = load_eigs(sample_name, f"{srr}_full")
        inter_sum, inter_count, strength = compute_saddle(clr, full_eig, view, task["nproc"])
        save_npz(out, interaction_sum=inter_sum, interaction_count=inter_count,
                 comp_strength=np.array([strength]))
        return ("done", entity["entity_id"])
    except Exception as error:
        return ("failed: " + repr(error)[:70], entity["entity_id"])
    finally:
        gc.collect()


fixed_tasks = [dict(sample_name=s, srr=r, entity=e, nproc=NPROC)
               for s, r in LIBRARIES
               for e in list_entities(s, r) if e["role"] == "subsample"]
run_parallel(fixed_tasks, worker_saddle_fixed, N_WORKERS, "saddle fullE1")

rows = []
for s, r in LIBRARIES:
    full_strength = load_saddle(s, f"{r}_full")["strength"]
    for e in list_entities(s, r):
        if e["role"] != "subsample" or not is_cached(saddle_fixed_path(s, e["entity_id"])):
            continue
        with np.load(saddle_fixed_path(s, e["entity_id"])) as d:
            fixed = float(d["comp_strength"][0])
        own = load_saddle(s, e["entity_id"])["strength"]
        rows.append(dict(sample_name=s, srr=r, depth=e["depth"],
                         own_E1=100 * own / full_strength,
                         full_E1=100 * fixed / full_strength))

control = pd.DataFrame(rows)
save_parquet(control, DATASETS_DIR / "saddle_fullE1_control.parquet")
display(control.groupby("depth")[["own_E1", "full_E1"]].agg(["mean", "std"]).round(1))

[1/756] saddle fullE1 skip         SRR13601502_100000000_sample2
[2/756] saddle fullE1 skip         SRR13601502_100000000_sample4
[3/756] saddle fullE1 skip         SRR13601502_10000000_sample1
[4/756] saddle fullE1 skip         SRR13601502_10000000_sample14
[5/756] saddle fullE1 skip         SRR13601502_10000000_sample12
[6/756] saddle fullE1 skip         SRR13601502_100000000_sample3
[7/756] saddle fullE1 skip         SRR13601502_10000000_sample13
[8/756] saddle fullE1 skip         SRR13601502_10000000_sample10
[9/756] saddle fullE1 skip         SRR13601502_10000000_sample11
[10/756] saddle fullE1 skip         SRR13601502_10000000_sample17
[11/756] saddle fullE1 skip         SRR13601502_10000000_sample16
[12/756] saddle fullE1 skip         SRR13601502_10000000_sample18
[13/756] saddle fullE1 skip         SRR13601502_10000000_sample20
[14/756] saddle fullE1 skip         SRR13601502_100000000_sample1
[15/756] saddle fullE1 skip         SRR13601502_10000000_sample19
[16/756] saddle full

own_E1       full_E1     
            mean   std    mean  std
depth                              
1000000     15.3  14.6   103.1  5.4
10000000   102.5  20.9   101.0  1.5
25000000   110.7   4.4   100.4  0.9
50000000   107.9   2.3   100.2  0.8
100000000  103.8   1.2   100.3  0.5
200000000  101.2   0.6   100.1  0.2